In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error
)
from sklearn.mixture import GaussianMixture
from tqdm import trange, tqdm
from kan import KAN
import copy

# === Config ===
SAMPLES      = 15000
TOT_EPOCHS   = 2000
KFOLD_SPLITS = 5
target_name = "s_n"
TARGETS      = ["s_n"]
GMM_ALPHA    = 0.8 # good for s_p 
GMM_ALPHA    = 2
VAL_FRAC     = 0.2
PATIENCE     = 80
WD           = 1e-4     # weight decay
LR           = 1e-3
GRID         = 12



RESULTS_DIR = "results"

# === Load & preprocess data ===
raw = torch.load("train_embeddings_250epochs_embdim128.pt").numpy()
mask = (raw[:, -3:] > 0).all(axis=1)
data = raw[mask]
np.random.seed(42)
idx = np.random.choice(data.shape[0], SAMPLES, replace=False)
data = data[idx]




X = data[:, :-3]
y_dict = {
    "band_gap": data[:, -3],
    "s_p":      data[:, -2],
    "s_n":      data[:, -1],
}

os.makedirs("results", exist_ok=True)

# scale X
X_scaler = StandardScaler()
X_scaled = X_scaler.fit_transform(X)
joblib.dump(X_scaler, "results/X_scaler.pkl")

# network architecture

WIDTH = [X.shape[1], 16, 1] ### Test for band Gap and Sn



# === Custom loss ===
class RelativeMSELoss(torch.nn.Module):
    def __init__(self, eps=1e-3):
        super().__init__()
        self.eps = eps
    def forward(self, pred, targ):
        return torch.mean(((pred - targ)/(torch.abs(targ)+self.eps))**2)



def train_with_cv(X, y_raw, name, model=None):
    print(f"\n▶️  Training: {name}")

    # --- scale target ---
    if name == "band_gap":
        scaler = StandardScaler()
        y_scaled = scaler.fit_transform(y_raw.reshape(-1,1)).flatten()
    elif name == "s_p":
        rs = RobustScaler()
        y_rs = rs.fit_transform(y_raw.reshape(-1,1)).flatten()
        pt = PowerTransformer(method='yeo-johnson')
        y_pt = pt.fit_transform(y_rs.reshape(-1,1)).flatten()
        gmm = GaussianMixture(2, random_state=42)
        probs = gmm.fit_predict(y_pt.reshape(-1,1))
        y_soft = y_pt + GMM_ALPHA*probs
        scaler = StandardScaler()
        y_scaled = scaler.fit_transform(y_soft.reshape(-1,1)).flatten()
    elif name == "s_n":
        rs = RobustScaler()
        y_rs = rs.fit_transform(y_raw.reshape(-1,1)).flatten()
        pt = PowerTransformer(method='yeo-johnson')
        y_pt = pt.fit_transform(y_rs.reshape(-1,1)).flatten()
        gmm = GaussianMixture(2, random_state=42)
        probs = gmm.fit_predict(y_pt.reshape(-1,1))
        y_soft = y_pt + GMM_ALPHA*probs
        scaler = StandardScaler()
        y_scaled = scaler.fit_transform(y_soft.reshape(-1,1)).flatten()

    import matplotlib.pyplot as plt
    import seaborn as sns

    print(y_raw) 
    fig, axs = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

    sns.kdeplot(y_raw, ax=axs[0], fill=True, color="gray")
    axs[0].set_title("Raw")

    if name == "s_n" or name == "s_p":
        sns.kdeplot(y_rs, ax=axs[1], fill=True, color="orange")
        axs[1].set_title("RobustScaled")

        sns.kdeplot(y_pt, ax=axs[2], fill=True, color="blue")
        axs[2].set_title("PowerTransformed")

        sns.kdeplot(y_soft, ax=axs[3], fill=True, color="green")
        axs[3].set_title("GMM Smoothed")

    else:
        sns.kdeplot(y_scaled, ax=axs[1], fill=True, color="blue")
        axs[1].set_title("StandardScaled")
        axs[2].axis("off")
        axs[3].axis("off")

    sns.kdeplot(y_scaled, ax=axs[4], fill=True, color="red")
    axs[4].set_title("Final Scaled")

    for ax in axs:
        ax.set_xlabel("Target Value")
        ax.grid(True)

    fig.suptitle(f"{name} Target Distribution Across Pipeline", fontsize=16)
    plt.tight_layout()
    plt.savefig(f"results/{name}_target_distribution.png", dpi=300)
    plt.show()

    joblib.dump(scaler, f"results/{name}_scaler.pkl")

    kf = KFold(n_splits=KFOLD_SPLITS, shuffle=True, random_state=42)
    results = []
    best_overall_rmse = float("inf")
    best_overall_model = None  # Track the actual best model across all folds
    all_train = []
    all_test  = []

    for fold_idx, (tr_idx, te_idx) in enumerate(kf.split(X), start=1):
        # full train/test split
        X_tr_full, X_te = X[tr_idx], X[te_idx]
        y_tr_full, y_te = y_scaled[tr_idx], y_scaled[te_idx]

        # train/val split for early stopping
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_tr_full, y_tr_full,
            test_size=VAL_FRAC,
            random_state=42
        )

        # model + optimizer
        # if no model is provided, create a new one
        if model is None:
            print("🔄 Creating new model")
            current_model = KAN(width=WIDTH, grid=GRID)
        else:
            print("🔄 Using provided model architecture, creating fresh copy")
            # Create a fresh copy of the model architecture for this fold
            current_model = KAN(width=WIDTH, grid=GRID)

        loss_fn = (RelativeMSELoss() if name in ["s_n","s_p"]
                   else torch.nn.MSELoss())
        optimizer = torch.optim.Adam(
            current_model.parameters(), lr=LR, weight_decay=WD
        )

        best_val_rmse = float("inf")
        no_improve    = 0
        best_state    = None

        # epoch loop with early stopping
        for epoch in trange(1, TOT_EPOCHS+1, desc=f"Fold {fold_idx}", leave=False):
            current_model.train()
            optimizer.zero_grad()
            out_tr = current_model(torch.tensor(X_tr, dtype=torch.float32))
            loss   = loss_fn(
                out_tr,
                torch.tensor(y_tr, dtype=torch.float32).view(-1,1)
            )
            loss.backward()
            optimizer.step()

            # validation
            current_model.eval()
            with torch.no_grad():
                y_val_pred = current_model(torch.tensor(X_val, dtype=torch.float32)).numpy().flatten()
            val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state    = current_model.state_dict()
                no_improve    = 0
            else:
                no_improve += 1
                if no_improve >= PATIENCE:
                    break

        # restore best weights for this fold
        current_model.load_state_dict(best_state)
        current_model.eval()

        # predictions on full train and test
        with torch.no_grad():
            y_tr_pred = current_model(torch.tensor(X_tr_full, dtype=torch.float32)).numpy().flatten()
            y_te_pred = current_model(torch.tensor(X_te,     dtype=torch.float32)).numpy().flatten()

        # unscale
        y_tr_true = scaler.inverse_transform(y_tr_full.reshape(-1,1)).flatten()
        y_te_true = scaler.inverse_transform(y_te.reshape(-1,1)).flatten()
        y_tr_pred = scaler.inverse_transform(y_tr_pred.reshape(-1,1)).flatten()
        y_te_pred = scaler.inverse_transform(y_te_pred.reshape(-1,1)).flatten()

        all_train.append(pd.DataFrame({
            f"{name}_true": y_tr_true,
            f"{name}_pred": y_tr_pred,
            "fold":         fold_idx
        }))
        all_test.append(pd.DataFrame({
            f"{name}_true": y_te_true,
            f"{name}_pred": y_te_pred,
            "fold":         fold_idx
        }))

        # compute metrics
        train_mse  = mean_squared_error(y_tr_true, y_tr_pred)
        train_rmse = np.sqrt(train_mse)
        train_mae  = mean_absolute_error(y_tr_true, y_tr_pred)
        train_r2   = r2_score(y_tr_true, y_tr_pred)

        test_mse   = mean_squared_error(y_te_true, y_te_pred)
        test_rmse  = np.sqrt(test_mse)
        test_mae   = mean_absolute_error(y_te_true, y_te_pred)
        test_r2    = r2_score(y_te_true, y_te_pred)

        print(f" Fold {fold_idx}  "
              f"Train → R²={train_r2:.3f}, RMSE={train_rmse:.3f}, "
              f"MAE={train_mae:.3f}, MSE={train_mse:.3f}  |  "
              f"Test  → R²={test_r2:.3f}, RMSE={test_rmse:.3f}, "
              f"MAE={test_mae:.3f}, MSE={test_mse:.3f}")

        # save best model overall (FIXED: now properly tracks the best model)
        if test_rmse < best_overall_rmse:
            best_overall_rmse = test_rmse
            best_overall_model = copy.deepcopy(current_model)  # Store the actual best model

            os.makedirs("results", exist_ok=True)
            os.makedirs("model", exist_ok=True)

            # 1) Lightweight weights (optionally include fold in filename)
            torch.save(current_model.state_dict(), f"results/{name}_KAN_best_model.pth")

            # 2) Full pykan checkpoint (arch + weights + meta [+optim])
            from datetime import datetime
            ts = datetime.now().strftime("%Y%m%d-%H%M%S")
            ckpt_prefix = f"./model/{name}_best_fold{fold_idx}_best"

            current_model.saveckpt(ckpt_prefix)

            print(
                f"✅ Saved best {name} so far | fold={fold_idx} | RMSE={test_rmse:.4f}\n"
                f"   - state_dict → results/{name}_KAN_best_model.pth\n"
                f"   - pykan ckpt  → {ckpt_prefix}.[meta|pth|optim]"
            )

        # collect for CSV
        results.append({
            "target":      name,
            "fold":        fold_idx,
            "train_r2":    train_r2,
            "train_mae":   train_mae,
            "train_rmse":  train_rmse,
            "train_mse":   train_mse,
            "test_r2":     test_r2,
            "test_mae":    test_mae,
            "test_rmse":   test_rmse,
            "test_mse":    test_mse,
        })

    pd.concat(all_train, ignore_index=True) \
      .to_csv(f"results/{name}_train_predictions.csv", index=False)
    pd.concat(all_test, ignore_index=True) \
      .to_csv(f"results/{name}_test_predictions.csv", index=False)
    # save fold‐level CSV
    pd.DataFrame(results).to_csv(f"results/{name}_fold_metrics.csv", index=False)
    # === Pair plots of true vs predicted ===
    for tgt in TARGETS:
        df_tr = pd.read_csv(f"results/{tgt}_train_predictions.csv")
        df_tr['split'] = 'train'
        df_te = pd.read_csv(f"results/{tgt}_test_predictions.csv")
        df_te['split'] = 'test'
        df_pp = pd.concat([df_tr, df_te], ignore_index=True)
        df_pp = df_pp.rename(columns={
            f"{tgt}_true": "True",
            f"{tgt}_pred": "Pred"
        })

        g = sns.pairplot(
            df_pp,
            vars=["True", "Pred"],
            hue="split",
            corner=True,
            diag_kind="kde",
            palette={'train': '#2a7044', 'test': '#a3471b'},
            plot_kws={'alpha':0.7, 'edgecolor':'k', 'linewidth':0.5}
        )
        g.fig.suptitle(f"Pair Plot – {tgt}", y=1.02, fontsize=16)
        plt.tight_layout()
        plt.savefig(f"results/{tgt}_pairplot.png", dpi=300, bbox_inches='tight')
        plt.show()

    print("✅ All metrics saved and pair plots generated.")
    return results, best_overall_model  # Return the truly best model across all folds


## **Train 1**

In [ ]:


# === Train all targets ===
all_metrics = []
best_models = {}

for tgt in TARGETS:
    results, model = train_with_cv(X_scaled, y_dict[tgt], tgt, model=None)
    all_metrics += results               # extend metrics list
    best_models[tgt] = model             # save model per target
    # y_scaled_last = y_scaled  # keep the scaled labels for this target

pd.DataFrame(all_metrics).to_csv(
    "results/KAN_all_cv_metrics_0.csv", index=False
)


In [ ]:
# ckpt_prefix = "./model/s_p_best_fold5"  # adjust as needed

# # Load model (weights + architecture + settings)
# model = KAN(width=WIDTH, grid=GRID)
# # model = model.loadckpt(ckpt_prefix)  # This loads weights and metadata
# import glob
# import os

# def get_latest_ckpt_prefix(target_name, model_dir=""):
#     ckpts = sorted(glob.glob(os.path.join(model_dir, "{target_name}_best_fold{fold_idx}_best")))
#     print(ckpts)
#     if not ckpts:
#         raise FileNotFoundError(f"No checkpoints found for {target_name} in {model_dir}")
#     # strip extension
#     return ckpts[-1].replace(".meta", "")

# # Example: load best model for s_p
# # prefix = get_latest_ckpt_prefix(")
# model = KAN.loadckpt('/Users/marco/Library/CloudStorage/Dropbox/Work/WorkingDirUSYD/Thermo/KANs_BangGap_Seebeck_tests/cf/Sp/model/s_p_best_fold5_20250825-183406')
# model.eval()
# print(f"✅ Loaded model from {prefix}")

In [ ]:
n_params = sum(p.numel() for p in best_models['s_n'].parameters())
print(f"Parameters: {n_params}")

In [ ]:
plt.figure(figsize=(12, 4))
best_models['s_n'].plot(beta=10)
plt.savefig(os.path.join(RESULTS_DIR, f"s_p_KAN_training_plot_trained_1.png"), dpi=900)
plt.show()

In [ ]:
from collections import defaultdict
import re

tot = 0
buckets = defaultdict(int)

for n, p in best_models['s_n'].named_parameters():
    cnt = p.numel()
    tot += cnt
    # crude bucketing by common substrings used in pykan-style modules
    if 'spline' in n or 'basis' in n or re.search(r'\b(c|coef|coeff)\b', n):
        buckets['spline_coeffs'] += cnt
    elif re.search(r'\bsp\b', n):
        buckets['edge_gate_sp'] += cnt
    elif re.search(r'\bsb\b', n):
        buckets['edge_gate_sb'] += cnt
    elif 'affine' in n or re.search(r'(scale|shift|a_?ff)', n):
        buckets['affine'] += cnt
    elif 'bias' in n:
        buckets['bias'] += cnt
    elif 'grid' in n or 'knot' in n:
        buckets['grid/knots'] += cnt
    else:
        buckets['other'] += cnt

print(f"Total parameters: {tot}")
for k, v in buckets.items():
    print(f"{k:15s}: {v}")

In [ ]:
from collections import defaultdict

param_details = []  # will store (name, type, count)

for n, p in best_models['s_n'].named_parameters():
    cnt = p.numel()

    # --- classify parameter type ---
    if "spline" in n or "basis" in n or "coef" in n or "coeff" in n:
        ptype = "spline_coeff (edge)"
    elif "sp" in n:
        ptype = "edge_gate_sp (edge)"
    elif "sb" in n:
        ptype = "edge_gate_sb (edge)"
    elif "affine" in n or "scale" in n or "shift" in n:
        ptype = "affine (edge)"
    elif "bias" in n:
        ptype = "bias (node)"
    elif "grid" in n or "knot" in n:
        ptype = "grid/knots (edge)"
    else:
        ptype = "other"

    param_details.append((n, ptype, cnt))

# --- print results nicely ---
total = 0
for name, ptype, cnt in param_details:
    print(f"{name:40s} | {ptype:20s} | {cnt}")
    total += cnt

print(f"\nTotal parameters: {total}")

In [ ]:
import re, math, numpy as np
from collections import defaultdict, Counter

# --- 0) Peek at names to learn the scheme ---
all_names = [n for n, _ in best_models['s_n'].named_parameters()]
print("Sample parameter names:")
for n in all_names[:25]:
    print("  ", n)
print(f"... total params listed: {len(all_names)}\n")

# --- 1) Patterns: handle dots, underscores, and brackets, with/without layer ---
# e.g. layers.0.edges.12., layer0.edge_12., edges[12]., edges.12., edge12., nodes.7., node_7.
EDGE_PATTERNS = [
    re.compile(r'(?:layers?\D*|layer\D*)?(\d+)?[._]?(?:edges?|edge)[\.\[_-](\d+)', re.I),
    re.compile(r'(?:edges?|edge)\[(\d+)\]', re.I),
    re.compile(r'(?:layers?\D*|layer\D*)?(\d+)?[._-]?(?:edges?|edge)(\d+)\b', re.I),  # edge12
    re.compile(r'(?:edges?|edge)[\._-](\d+)', re.I),
]
NODE_PATTERNS = [
    re.compile(r'(?:layers?\D*|layer\D*)?(\d+)?[._]?(?:nodes?|node)[\.\[_-](\d+)', re.I),
    re.compile(r'(?:nodes?|node)\[(\d+)\]', re.I),
    re.compile(r'(?:layers?\D*|layer\D*)?(\d+)?[._-]?(?:nodes?|node)(\d+)\b', re.I),  # node7
    re.compile(r'(?:nodes?|node)[\._-](\d+)', re.I),
]

def parse_edge(name: str):
    for pat in EDGE_PATTERNS:
        m = pat.search(name)
        if m:
            # group(1) may be layer (optional), group(2) is edge id; if only one group, it's the id
            if m.lastindex == 2:
                layer = m.group(1)
                return (int(layer) if layer and layer.isdigit() else None, int(m.group(2)))
            else:
                return (None, int(m.group(1)))
    return None

def parse_node(name: str):
    for pat in NODE_PATTERNS:
        m = pat.search(name)
        if m:
            if m.lastindex == 2:
                layer = m.group(1)
                return (int(layer) if layer and layer.isdigit() else None, int(m.group(2)))
            else:
                return (None, int(m.group(1)))
    return None

# --- 2) Collect stats ---
edges = defaultdict(lambda: dict(n_params=0, spline_L1=0.0, sp_L1=0.0, sb_L1=0.0, affine_L1=0.0, grid_cnt=0, other_cnt=0))
nodes = defaultdict(lambda: dict(bias_cnt=0, bias_L1=0.0, fan_in=0, fan_out=0))

def add_L1(d, name, tensor):
    t = tensor.detach().float().view(-1)
    v = t.abs().sum().item()
    if "spline" in name or "basis" in name or "coef" in name or "coeff" in name:
        d["spline_L1"] += v
    elif re.search(r'\bsp\b', name):
        d["sp_L1"] += v
    elif re.search(r'\bsb\b', name):
        d["sb_L1"] += v
    elif ("affine" in name) or ("scale" in name) or ("shift" in name) or ("a_ff" in name):
        d["affine_L1"] += v
    elif ("grid" in name) or ("knot" in name):
        d["grid_cnt"] += tensor.numel()
    else:
        d["other_cnt"] += tensor.numel()

matched_edges = matched_nodes = 0
for n, p in best_models['s_n'].named_parameters():
    lname = n.lower()
    ek = parse_edge(lname)
    nk = parse_node(lname)

    if ek is not None:
        E = edges[ek]
        E["n_params"] += p.numel()
        add_L1(E, lname, p)
        matched_edges += 1
    elif nk is not None:
        N = nodes[nk]
        if "bias" in lname:
            N["bias_cnt"] += p.numel()
            N["bias_L1"] += p.detach().float().abs().sum().item()
        matched_nodes += 1

print(f"Matched edge-like params: {matched_edges}, node-like params: {matched_nodes}")

# --- 3) Fallback if STILL empty: group by parent module path ---
if len(edges) == 0 and len(nodes) == 0:
    print("\n[Fallback] No edges/nodes matched. Showing top parent buckets (by prefix before last '.'):")
    parents = defaultdict(int)
    for n, p in best_models['s_n'].named_parameters():
        parent = n.rsplit('.', 1)[0] if '.' in n else n
        parents[parent] += p.numel()
    top = sorted(parents.items(), key=lambda x: x[1], reverse=True)[:30]
    for k, v in top:
        print(f"  {k:50s}  {v}")
    # you can identify which parents correspond to edges/nodes and tweak patterns above

# --- 4) Scores & ranking ---
def edge_score(E):
    return 1.0*E["spline_L1"] + 0.5*(E["sp_L1"] + E["sb_L1"]) + 0.25*E["affine_L1"]

def node_score(N):
    return 0.5*N["fan_in"] + 0.5*N["bias_L1"]

ranked_edges = sorted(((k, edge_score(v), v) for k, v in edges.items()), key=lambda x: x[1], reverse=True)
ranked_nodes = sorted(((k, node_score(v), v) for k, v in nodes.items()), key=lambda x: x[1], reverse=True)

print("\nTop edges (by strength):")
for (layer, eidx), sc, v in ranked_edges[:20]:
    print(f"  layer={layer}, edge={eidx} | score={sc:.3e} | "
          f"spline_L1={v['spline_L1']:.3e}, sp_L1={v['sp_L1']:.3e}, sb_L1={v['sb_L1']:.3e}, "
          f"affine_L1={v['affine_L1']:.3e}, n_params={v['n_params']}")

print("\nTop nodes (by importance):")
for (layer, nidx), sc, v in ranked_nodes[:20]:
    print(f"  layer={layer}, node={nidx} | score={sc:.3e} | bias_L1={v['bias_L1']:.3e}, bias_cnt={v['bias_cnt']}")

In [ ]:
from collections import defaultdict
import torch
import math

edges = defaultdict(lambda: dict(n_params=0, coef_L1=0.0, grid_cnt=0,
                                 sp_L1=0.0, base_L1=0.0, mask_cnt=0, affine_L1=0.0))
nodes = defaultdict(lambda: dict(bias_L1=0.0, scale_L1=0.0))

for n, p in best_models['s_n'].named_parameters():
    pname = n.lower()
    cnt = p.numel()

    # --- Edges / activations ---
    if pname.startswith("act_fun."):
        # extract edge id (act_fun.j.xxx)
        eid = int(pname.split(".")[1])
        E = edges[eid]
        E["n_params"] += cnt

        if "coef" in pname:
            E["coef_L1"] += p.detach().float().abs().sum().item()
        elif "grid" in pname:
            E["grid_cnt"] += cnt
        elif "scale_sp" in pname:
            E["sp_L1"] += p.detach().float().abs().sum().item()
        elif "scale_base" in pname:
            E["base_L1"] += p.detach().float().abs().sum().item()
        elif "mask" in pname:
            E["mask_cnt"] += cnt

    # --- Symbolic functions (treated like edges) ---
    elif pname.startswith("symbolic_fun."):
        eid = int(pname.split(".")[1])
        E = edges[f"sym_{eid}"]
        E["n_params"] += cnt
        if "affine" in pname:
            E["affine_L1"] += p.detach().float().abs().sum().item()
        elif "mask" in pname:
            E["mask_cnt"] += cnt

    # --- Nodes ---
    elif pname.startswith("node_bias"):
        nid = int(pname.split("_")[-1])
        nodes[nid]["bias_L1"] += p.detach().float().abs().sum().item()
    elif pname.startswith("node_scale"):
        nid = int(pname.split("_")[-1])
        nodes[nid]["scale_L1"] += p.detach().float().abs().sum().item()
    elif pname.startswith("subnode_bias"):
        nid = int(pname.split("_")[-1])
        nodes[f"sub{nid}"]["bias_L1"] += p.detach().float().abs().sum().item()
    elif pname.startswith("subnode_scale"):
        nid = int(pname.split("_")[-1])
        nodes[f"sub{nid}"]["scale_L1"] += p.detach().float().abs().sum().item()

# --- Define importance scores ---
def edge_score(E):
    return E["coef_L1"] + 0.5*E["sp_L1"] + 0.5*E["base_L1"] + 0.25*E["affine_L1"]

def node_score(N):
    return N["bias_L1"] + 0.5*N["scale_L1"]

ranked_edges = sorted(edges.items(), key=lambda kv: edge_score(kv[1]), reverse=True)
ranked_nodes = sorted(nodes.items(), key=lambda kv: node_score(kv[1]), reverse=True)

print("\nTop edges (by strength):")
for eid, v in ranked_edges[:20]:
    print(f" edge={eid} | score={edge_score(v):.3f} "
          f"| coef={v['coef_L1']:.2f}, sp={v['sp_L1']:.2f}, base={v['base_L1']:.2f}, "
          f"affine={v['affine_L1']:.2f}, params={v['n_params']}")

print("\nTop nodes (by importance):")
for nid, v in ranked_nodes[:20]:
    print(f" node={nid} | score={node_score(v):.3f} "
          f"| bias={v['bias_L1']:.2f}, scale={v['scale_L1']:.2f}")

In [ ]:
edges_to_extract = ["act_fun.0"]      # strongest learned edge
maybe_edges      = ["act_fun.1"]      # optional, weaker
exclude_edges    = ["symbolic_fun.0"] # too dominant / already symbolic
keep_symbolic    = ["symbolic_fun.1"] # okay to keep if needed

In [ ]:
# Check masks and magnitudes for the edge/basis you care about
for key in ["act_fun.0.mask", "act_fun.0.coef", "act_fun.0.scale_sp", "act_fun.0.scale_base"]:
    t = dict(best_models['s_n'].named_parameters()).get(key, None)
    if t is not None:
        t = t.detach().float()
        print(f"{key:20s} | L1={t.abs().sum().item():.3e} | nonzeros={(t!=0).sum().item()}")
# Check masks and magnitudes for the edge/basis you care about
for key in ["act_fun.1.mask", "act_fun.1.coef", "act_fun.1.scale_sp", "act_fun.1.scale_base"]:
    t = dict(best_models['s_n'].named_parameters()).get(key, None)
    if t is not None:
        t = t.detach().float()
        print(f"{key:20s} | L1={t.abs().sum().item():.3e} | nonzeros={(t!=0).sum().item()}")

In [ ]:
expr = best_models['s_n'].auto_symbolic(
    a_range=(-20, 20), b_range=(-20, 20),  # wider than default
    weight_simple=0.2,                     # less simplicity pressure
    r2_threshold=0.1,                      # don’t reject for low R^2
    verbose=2
)

In [ ]:
# ============================================
# KAN symbolic extraction: focused & robust
# ============================================
# - Ranks edges/nodes from parameter magnitudes
# - Selects top-K meaningful edges
# - Masks out others (safely) and calls auto_symbolic
# - Falls back gracefully if some kwargs unsupported
# - Optional: hooks to check variance on a chosen edge
#
# Works with param names like:
#   act_fun.0.grid / .coef / .mask / .scale_base / .scale_sp
#   symbolic_fun.0.mask / .affine
#   node_bias_0, node_scale_0, subnode_bias_0, subnode_scale_0
# ============================================

import contextlib, inspect, math, torch
from collections import defaultdict


def _supports_kwargs(fn, **kwargs):
    """Return only kwargs that are accepted by fn."""
    try:
        sig = inspect.signature(fn)
        return {k: v for k, v in kwargs.items() if k in sig.parameters}
    except (TypeError, ValueError):
        return {}

def _l1(t):
    return float(t.detach().float().abs().sum().item())

def _nonzeros(t):
    return int((t.detach() != 0).sum().item())

def sample_param_names(model, n=25):
    print("Sample parameter names:")
    for i, (nme, _) in enumerate(model.named_parameters()):
        if i >= n: break
        print(" ", nme)
    print()


def rank_edges_nodes(model):
    """Rank edges (act_fun.*, symbolic_fun.*) and nodes by importance."""
    edges = defaultdict(lambda: dict(
        n_params=0, coef_L1=0.0, sp_L1=0.0, base_L1=0.0,
        affine_L1=0.0, grid_cnt=0, mask_cnt=0
    ))
    nodes = defaultdict(lambda: dict(bias_L1=0.0, scale_L1=0.0))

    for nme, p in model.named_parameters():
        lname = nme.lower()
        cnt = p.numel()

        if lname.startswith("act_fun."):
            eid = int(lname.split(".")[1])
            E = edges[f"act_{eid}"]
            E["n_params"] += cnt
            if ".coef" in lname:
                E["coef_L1"] += _l1(p)
            elif ".scale_sp" in lname:
                E["sp_L1"] += _l1(p)
            elif ".scale_base" in lname:
                E["base_L1"] += _l1(p)
            elif ".affine" in lname:
                E["affine_L1"] += _l1(p)
            elif ".grid" in lname:
                E["grid_cnt"] += cnt
            elif ".mask" in lname:
                E["mask_cnt"] += cnt

        elif lname.startswith("symbolic_fun."):
            eid = int(lname.split(".")[1])
            E = edges[f"sym_{eid}"]
            E["n_params"] += cnt
            if ".affine" in lname:
                E["affine_L1"] += _l1(p)
            elif ".mask" in lname:
                E["mask_cnt"] += cnt

        # Nodes
        elif lname.startswith("node_bias_"):
            nid = lname.split("_")[-1]
            nodes[f"node_{nid}"]["bias_L1"] += _l1(p)
        elif lname.startswith("node_scale_"):
            nid = lname.split("_")[-1]
            nodes[f"node_{nid}"]["scale_L1"] += _l1(p)
        elif lname.startswith("subnode_bias_"):
            nid = lname.split("_")[-1]
            nodes[f"subnode_{nid}"]["bias_L1"] += _l1(p)
        elif lname.startswith("subnode_scale_"):
            nid = lname.split("_")[-1]
            nodes[f"subnode_{nid}"]["scale_L1"] += _l1(p)

    def edge_score(E):
        return E["coef_L1"] + 0.5*(E["sp_L1"] + E["base_L1"]) + 0.25*E["affine_L1"]

    def edge_score_norm(E):
        return edge_score(E) / max(1, E["n_params"])

    def node_score(N):
        return N["bias_L1"] + 0.5*N["scale_L1"]

    ranked_edges = sorted(edges.items(), key=lambda kv: edge_score(kv[1]), reverse=True)
    ranked_edges_norm = sorted(edges.items(), key=lambda kv: edge_score_norm(kv[1]), reverse=True)
    ranked_nodes = sorted(nodes.items(), key=lambda kv: node_score(kv[1]), reverse=True)

    return ranked_edges, ranked_edges_norm, ranked_nodes, edges, nodes


@contextlib.contextmanager
def keep_only_prefixes(model, include_prefixes):
    """
    Zero *other* .mask tensors so only selected blocks contribute.
    Leaves included blocks' internal mask values untouched.
    """
    saved = {}
    try:
        with torch.no_grad():
            for nme, p in model.named_parameters():
                if nme.endswith(".mask"):
                    saved[nme] = p.detach().clone()
                    if not any(nme.startswith(pref) for pref in include_prefixes):
                        p.zero_()
        yield
    finally:
        with torch.no_grad():
            for nme, p in model.named_parameters():
                if nme in saved:
                    p.copy_(saved[nme])


def check_edge_health(model, act_ids):
    """Print L1 and nonzeros for masks/coef/scales of given act_fun IDs."""
    pd = dict(model.named_parameters())
    for j in act_ids:
        for key in [f"act_fun.{j}.mask",
                    f"act_fun.{j}.coef",
                    f"act_fun.{j}.scale_sp",
                    f"act_fun.{j}.scale_base"]:
            t = pd.get(key, None)
            if t is not None:
                print(f"{key:22s} | L1={_l1(t):.3e} | nz={_nonzeros(t)}")
            else:
                print(f"{key:22s} | <missing>")


def variance_on_act_fun(model, j, dataloader=None, device=None, max_batches=5):
    """
    Registers a forward hook on model.act_fun[j] (if present),
    runs a few batches, and returns std(u), std(v).
    """
    act_mod = None
    if hasattr(model, "act_fun"):
        try:
            act_mod = model.act_fun[j]
        except Exception:
            act_mod = None
    if act_mod is None:
        print(f"[warn] model.act_fun[{j}] not found; skipping variance check.")
        return None, None

    U, V = [], []

    def hook(_, inputs, output):
        u = inputs[0].detach().flatten().cpu()
        v = output.detach().flatten().cpu()
        U.append(u); V.append(v)

    handle = act_mod.register_forward_hook(hook)

    model.eval()
    with torch.no_grad():
        if dataloader is None:
            print("[info] No dataloader given; variance check skipped.")
            handle.remove()
            return None, None
        for bi, batch in enumerate(dataloader):
            if isinstance(batch, (list, tuple)):
                xb = batch[0]
            else:
                xb = batch
            if device is not None:
                xb = xb.to(device)
            _ = model(xb)
            if bi+1 >= max_batches:
                break

    handle.remove()
    import numpy as np
    u = torch.cat(U) if len(U) else torch.tensor([])
    v = torch.cat(V) if len(V) else torch.tensor([])
    ustd = float(u.std().item()) if u.numel() else 0.0
    vstd = float(v.std().item()) if v.numel() else 0.0
    print(f"[variance] act_fun[{j}]: std(u)={ustd:.3e}, std(v)={vstd:.3e}")
    return ustd, vstd


def call_auto_symbolic(model, **kwargs):

    if not hasattr(model, "auto_symbolic"):
        raise AttributeError("model has no auto_symbolic method")

    defaults = dict(
        weight_simple=0.0,
        r2_threshold=0.0,
        lib=["x", "x^2", "sin", "cos"],
        a_range=(-20, 20),
        b_range=(-20, 20),
        verbose=1,
    )
    defaults.update(kwargs or {})
    filtered = _supports_kwargs(model.auto_symbolic, **defaults)
    return model.auto_symbolic(**filtered)


def extract_symbols_focused(model, dataloader=None, device=None, top_k=2,
                            exclude_prefixes=("symbolic_fun.0",),  # common big head to exclude
                            attempt_settings=None):
    """
    1) Rank edges/nodes
    2) Choose top-K act_fun edges (normalize-aware)
    3) Check health & variance (optional)
    4) Mask others, call auto_symbolic with robust settings
    """
    print("=== Inspect names ===")
    sample_param_names(model, n=20)

    print("=== Rank edges/nodes ===")
    ranked_edges, ranked_edges_norm, ranked_nodes, edges_map, nodes_map = rank_edges_nodes(model)

    print("\nTop edges (raw score):")
    for k, v in ranked_edges[:10]:
        print(f"  {k:10s} | n_params={v['n_params']:6d} | "
              f"coef={v['coef_L1']:.2f}, sp={v['sp_L1']:.2f}, base={v['base_L1']:.2f}, affine={v['affine_L1']:.2f}")

    print("\nTop edges (normalized by params):")
    def _norm_score(v):
        return (v["coef_L1"] + 0.5*(v["sp_L1"]+v["base_L1"]) + 0.25*v["affine_L1"]) / max(1, v["n_params"])
    for k, v in ranked_edges_norm[:10]:
        print(f"  {k:10s} | score_norm={_norm_score(v):.4f} | n_params={v['n_params']}")

    print("\nTop nodes:")
    for k, v in ranked_nodes[:10]:
        print(f"  {k:10s} | bias={v['bias_L1']:.2f}, scale={v['scale_L1']:.2f}")

    act_edges_sorted = [k for k, _ in ranked_edges_norm if k.startswith("act_")]
    chosen = act_edges_sorted[:top_k]
    print(f"\nChosen act_fun edges for extraction (top_k={top_k}): {chosen}")

    act_ids = [int(k.split("_")[1]) for k in chosen]
    print("\n=== Edge health (masks & magnitudes) ===")
    check_edge_health(model, act_ids)

    if dataloader is not None:
        print("\n=== Variance checks on chosen edges ===")
        for j in act_ids:
            variance_on_act_fun(model, j, dataloader=dataloader, device=device)

    if attempt_settings is None:
        attempt_settings = [
            dict(weight_simple=0.0, r2_threshold=0.0, lib=["x","x^2","sin","cos"]),   # fit-first
            dict(weight_simple=0.2, r2_threshold=0.0, lib=["x","x^2","sin","cos"]),
            dict(weight_simple=0.6, r2_threshold=0.05, lib=["x","sin","cos"]),        # encourage simplicity
            dict(weight_simple=0.2, r2_threshold=0.1, lib=["x^2","sin","cos"]),       # nudge away from identity
        ]

    for k in chosen:
        j = int(k.split("_")[1])
        include = [f"act_fun.{j}"]
        print(f"\n=== Extracting symbols for {k} (include={include}, exclude={exclude_prefixes}) ===")

        with keep_only_prefixes(model, include_prefixes=include):
            with torch.no_grad():
                for nme, p in model.named_parameters():
                    if nme.endswith(".mask") and any(nme.startswith(pref) for pref in exclude_prefixes):
                        p.zero_()

            expr = None
            for i, sett in enumerate(attempt_settings, 1):
                print(f"  [Attempt {i}] settings={sett}")
                expr = call_auto_symbolic(model, **sett)
                print("  -> expr:", expr)

            print(f"[Result for {k}] last expr:", expr)

    print("\n=== Done ===")



In [ ]:
best_models['s_n'].eval()
extract_symbols_focused(best_models['s_n'], dataloader=None, device=None, top_k=2,
                        exclude_prefixes=("symbolic_fun.0",))

extract_symbols_focused(model1, top_k=2)

In [ ]:
best_models['s_n'].eval()
expr = best_models['s_n'].auto_symbolic(
    weight_simple=0.0,   # allow complex if needed
    r2_threshold=0.0,    # don’t reject
    lib=["x^2","sin","cos", "exp", "log"],  # compact but expressive
)
print(expr)

In [ ]:
import re
import math
import torch
import numpy as np
from collections import defaultdict, Counter


EDGE_RE = re.compile(r'(layer|layers)\.?(\d+)\.?(edges|edge)\.?(\d+)', re.I)
ALT_EDGE_RE = re.compile(r'(edges?|edge)\.?(\d+)', re.I)  # fallback if no layer in name
NODE_RE = re.compile(r'(layer|layers)\.?(\d+)\.?(nodes?|node)\.?(\d+)', re.I)
ALT_NODE_RE = re.compile(r'(nodes?|node)\.?(\d+)', re.I)

def parse_edge_key(name):
    m = EDGE_RE.search(name)
    if m:
        layer = int(m.group(2)); edge = int(m.group(4))
        return (layer, edge)
    m = ALT_EDGE_RE.search(name)
    if m:
        return (None, int(m.group(2)))
    return None

def parse_node_key(name):
    m = NODE_RE.search(name)
    if m:
        layer = int(m.group(2)); node = int(m.group(4))
        return (layer, node)
    m = ALT_NODE_RE.search(name)
    if m:
        return (None, int(m.group(2)))
    return None

edges = defaultdict(lambda: {
    "n_params": 0,
    "spline_L1": 0.0,
    "spline_L2": 0.0,
    "sp_L1": 0.0,
    "sb_L1": 0.0,
    "affine_L1": 0.0,
    "grid_cnt": 0,
    "other_cnt": 0,
})
nodes = defaultdict(lambda: {
    "bias_cnt": 0,
    "bias_L1": 0.0,
    "fan_in": 0,    # we’ll infer after we know edge -> dst mapping (best-effort)
    "fan_out": 0,   # idem, for src (if available)
})

DST_RE = re.compile(r'\b(to|dst|dest|out|target)\.?(?P<dst>\d+)', re.I)
SRC_RE = re.compile(r'\b(from|src|in|source)\.?(?P<src>\d+)', re.I)

edge2dst = {}
edge2src = {}

def add_norm(stats, tensor, key_prefix):
    t = tensor.detach().float().cpu().view(-1)
    if t.numel() == 0: return
    if key_prefix == "spline":
        stats["spline_L1"] += t.abs().sum().item()
        stats["spline_L2"] += math.sqrt(float((t*t).sum().item()))
    elif key_prefix == "sp":
        stats["sp_L1"] += t.abs().sum().item()
    elif key_prefix == "sb":
        stats["sb_L1"] += t.abs().sum().item()
    elif key_prefix == "affine":
        stats["affine_L1"] += t.abs().sum().item()

for n, p in best_models['s_n'].named_parameters():
    cnt = p.numel()
    ek = parse_edge_key(n)
    nk = parse_node_key(n)

    md = DST_RE.search(n)
    ms = SRC_RE.search(n)
    if ek:
        if md: edge2dst[ek] = int(md.group("dst"))
        if ms: edge2src[ek] = int(ms.group("src"))

    if ek:
        E = edges[ek]
        E["n_params"] += cnt
        lname = n.lower()
        if ("spline" in lname) or ("basis" in lname) or ("coef" in lname) or ("coeff" in lname):
            add_norm(E, p, "spline")
        elif re.search(r'\bsp\b', n):
            add_norm(E, p, "sp")
        elif re.search(r'\bsb\b', n):
            add_norm(E, p, "sb")
        elif ("affine" in lname) or ("scale" in lname) or ("shift" in lname) or ("a_ff" in lname):
            add_norm(E, p, "affine")
        elif ("grid" in lname) or ("knot" in lname):
            E["grid_cnt"] += cnt
        else:
            E["other_cnt"] += cnt

    elif nk:
        N = nodes[nk]
        if "bias" in n.lower():
            N["bias_cnt"] += cnt
            N["bias_L1"] += p.detach().float().abs().sum().item()


dst_counter = Counter()
src_counter = Counter()
for ek in edges.keys():
    dst = edge2dst.get(ek, None)
    src = edge2src.get(ek, None)
    if dst is not None:
        dst_counter[(ek[0], dst)] += 1
    if src is not None:
        src_counter[(ek[0], src)] += 1

for nk in nodes.keys():
    nodes[nk]["fan_in"] = dst_counter.get(nk, nodes[nk]["fan_in"])
    nodes[nk]["fan_out"] = src_counter.get(nk, nodes[nk]["fan_out"])

# ---- Define importance scores ----
def edge_score(E):
    # you can tune the weights; this one balances spline magnitude + gates + affine
    return (
        1.0 * E["spline_L1"] +
        0.5 * (E["sp_L1"] + E["sb_L1"]) +
        0.25 * E["affine_L1"]
    )

def node_score(N):
    return 0.5 * N["fan_in"] + 0.5 * N["bias_L1"]

ranked_edges = sorted(
    [(k, edge_score(v), v) for k, v in edges.items()],
    key=lambda x: x[1], reverse=True
)
ranked_nodes = sorted(
    [(k, node_score(v), v) for k, v in nodes.items()],
    key=lambda x: x[1], reverse=True
)

TOP = 20
print("\nTop edges (by strength):")
for (layer, eidx), sc, v in ranked_edges[:TOP]:
    print(f"  layer={layer}, edge={eidx} | score={sc:.3f} | "
          f"spline_L1={v['spline_L1']:.2f}, sp_L1={v['sp_L1']:.2f}, sb_L1={v['sb_L1']:.2f}, "
          f"affine_L1={v['affine_L1']:.2f}, n_params={v['n_params']}")

print("\nTop nodes (by importance):")
for (layer, nidx), sc, v in ranked_nodes[:TOP]:
    print(f"  layer={layer}, node={nidx} | score={sc:.3f} | "
          f"fan_in={v['fan_in']}, bias_L1={v['bias_L1']:.2f}, bias_cnt={v['bias_cnt']}")

In [ ]:
best_model_edges = best_models['s_n'].auto_symbolic()

In [ ]:
best_models['s_n'].symbolic_formula()[:][:]


In [ ]:
import os
import numpy as np
import pandas as pd
import torch


def _l1_normalize(x, axis=None, eps=1e-12):
    x = np.nan_to_num(np.asarray(x, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    s = x.sum(axis=axis, keepdims=True) + eps
    return x / s

def _as_numpy(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu().numpy()
    return np.asarray(x)

def _ensure_list(obj):
    if obj is None:
        return []
    if isinstance(obj, (list, tuple)):
        return list(obj)
    return [obj]


def compute_kan_built_in_relevance(model,
                                   X_scaled: np.ndarray,
                                   results_dir: str = "results",
                                   batch_size: int = 2048,
                                   normalise: bool = True,
                                   save_csv: bool = True):

    os.makedirs(results_dir, exist_ok=True)
    device = next(model.parameters()).device
    model.eval()

    X_t = torch.tensor(X_scaled, dtype=torch.float32, device=device)
    with torch.no_grad():
        for s in range(0, X_t.shape[0], batch_size):
            xb = X_t[s:s+batch_size]
            try:
                _ = model.get_act(xb, record=True)
            except TypeError:
                _ = model.get_act(xb)

    ran = False
    for kwargs in ({"aggregate": "mean"}, {"aggregate": "sum"}, {}):
        try:
            model.attribute(**kwargs)
            ran = True
            break
        except TypeError:
            continue
    if not ran:
        raise RuntimeError("model.attribute(...) not available on this KAN build.")

    feat = _as_numpy(getattr(model, "feature_score", None))
    nodes = getattr(model, "node_score", None)
    edges = getattr(model, "edge_score", None)

    node_list = [_as_numpy(x) for x in _ensure_list(nodes)]
    edge_list = [_as_numpy(x) for x in _ensure_list(edges)]

    out = {
        "feature_score_raw": feat,
        "node_score_raw_list": node_list,
        "edge_score_raw_list": edge_list,
        "csv": {}
    }

    if feat is not None and normalise:
        feat_n = _l1_normalize(feat.ravel())
        out["feature_score"] = feat_n
    elif feat is not None:
        out["feature_score"] = feat

    normed_nodes = []
    for ns in node_list:
        if ns is None:
            normed_nodes.append(None)
            continue
        vec = ns.ravel()
        normed_nodes.append(_l1_normalize(vec))
    out["node_score_list"] = normed_nodes

    normed_edges = []
    for es in edge_list:
        if es is None:
            normed_edges.append(None)
            continue
        es_n = _l1_normalize(es)
        normed_edges.append(es_n)
    out["edge_score_list"] = normed_edges

    if save_csv:
        if feat is not None:
            df = pd.DataFrame({
                "feature_index": np.arange(out["feature_score"].size, dtype=int),
                "score": out["feature_score"]
            })
            p = os.path.join(results_dir, "kan_feature_score.csv")
            df.to_csv(p, index=False)
            out["csv"]["feature_score"] = p

        # node scores by layer (L=0..)
        for L, ns in enumerate(out["node_score_list"]):
            if ns is None:
                continue
            df = pd.DataFrame({
                "layer": L,
                "node_index": np.arange(ns.size, dtype=int),
                "score": ns
            })
            p = os.path.join(results_dir, f"kan_node_score_L{L}.csv")
            df.to_csv(p, index=False)
            out["csv"][f"node_score_L{L}"] = p

        for L, es in enumerate(out["edge_score_list"]):
            if es is None:
                continue
            arr = np.asarray(es)
            if arr.ndim != 2:
                arr = arr.reshape(arr.shape[0], -1)
            fan_in, fan_out = arr.shape[0], arr.shape[1]
            rows = {
                "layer": np.full(fan_in * fan_out, L, dtype=int),
                "in_idx": np.repeat(np.arange(fan_in, dtype=int), fan_out),
                "out_idx": np.tile(np.arange(fan_out, dtype=int), fan_in),
                "score": arr.ravel()
            }
            df = pd.DataFrame(rows)
            p = os.path.join(results_dir, f"kan_edge_score_L{L}.csv")
            df.to_csv(p, index=False)
            out["csv"][f"edge_score_L{L}"] = p

    def _shape(x):
        return None if x is None else tuple(x.shape)

    print("[KAN] feature_score shape:", _shape(feat))
    print("[KAN] node_score layers:", [None if n is None else n.shape for n in _ensure_list(nodes)])
    print("[KAN] edge_score layers:", [None if e is None else e.shape for e in _ensure_list(edges)])

    return out



In [ ]:

res = compute_kan_built_in_relevance(model, X_scaled, results_dir="results", batch_size=2048, normalise=True, save_csv=True)

In [ ]:


import os
import numpy as np
import pandas as pd
import torch


def _np(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().float().cpu().numpy()
    return np.asarray(x)

def _l1(x, eps=1e-12):
    x = np.nan_to_num(_np(x), nan=0.0, posinf=0.0, neginf=0.0)
    s = x.sum() + eps
    return x / s

def _ensure_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    return [x]

def _shape(x):
    try:
        return None if x is None else tuple(x.shape)
    except Exception:
        return None

def _print_shapes(tag, arr):
    if isinstance(arr, (list, tuple)):
        print(f"[{tag}] layers:", [ _shape(a) for a in arr ])
    else:
        print(f"[{tag}] shape:", _shape(arr))

# ------------------ main routine ------------------

def compute_kan_built_in_relevance_full(
    model,
    X_scaled: np.ndarray,
    results_dir: str = "results",
    batch_size: int = 2048,
    normalise: bool = True,
    save_csv: bool = True,
):
    os.makedirs(results_dir, exist_ok=True)
    device = next(model.parameters()).device
    model.eval()

    X_t = torch.tensor(X_scaled, dtype=torch.float32, device=device)
    with torch.no_grad():
        for s in range(0, X_t.shape[0], batch_size):
            xb = X_t[s:s+batch_size]
            try:
                _ = model.get_act(xb, record=True)
            except TypeError:
                _ = model.get_act(xb)

    attr_return = None
    tried = []
    for kwargs in (
        {"return_dict": True, "aggregate": "mean"},
        {"return_all": True,   "aggregate": "mean"},
        {"aggregate": "mean"},
        {"aggregate": "sum"},
        {},
    ):
        try:
            attr_return = model.attribute(**kwargs)
            print("[attribute] ran with kwargs =", kwargs)
            break
        except TypeError as e:
            tried.append(str(e))
            continue
        except Exception as e:
            print("[attribute] warning:", e)
            continue

    feat_score = None
    node_scores = []
    edge_scores = []

    if isinstance(attr_return, dict):
        # try common keys
        for k in ("feature_score", "features", "feature_importance"):
            if k in attr_return:
                feat_score = _np(attr_return[k])
                break
        for k in ("node_score", "node_scores", "nodes"):
            if k in attr_return:
                node_scores = _ensure_list(attr_return[k])
                break
        for k in ("edge_score", "edge_scores", "edges", "edge_importance"):
            if k in attr_return:
                edge_scores = _ensure_list(attr_return[k])
                break

    if feat_score is None and hasattr(model, "feature_score"):
        feat_score = _np(getattr(model, "feature_score"))
    if not node_scores:
        for name in ("node_score", "node_scores"):
            if hasattr(model, name):
                node_scores = _ensure_list(getattr(model, name))
                break
    if not edge_scores:
        for name in ("edge_score", "edge_scores", "edge_importance"):
            if hasattr(model, name):
                edge_scores = _ensure_list(getattr(model, name))
                break

    node_scores = [ _np(x) for x in node_scores if x is not None ]
    edge_scores = [ _np(x) for x in edge_scores if x is not None ]

    _print_shapes("feature_score", feat_score)
    _print_shapes("node_score", node_scores)
    _print_shapes("edge_score", edge_scores)

    # 5) Normalise (global L1)
    feat_norm = None
    if feat_score is not None:
        feat_norm = _l1(feat_score)

    node_norm_list = []
    for ns in node_scores:
        # per-layer node vector: flatten then L1
        node_norm_list.append(_l1(ns.ravel()))

    edge_norm_list = []
    for es in edge_scores:
        edge_norm_list.append(_l1(es))

    # 6) Save CSVs (tidy format)
    csv_paths = {}
    if save_csv:
        if feat_norm is not None:
            df = pd.DataFrame({
                "feature_index": np.arange(feat_norm.size, dtype=int),
                "score": feat_norm
            })
            p = os.path.join(results_dir, "kan_feature_score.csv")
            df.to_csv(p, index=False)
            csv_paths["feature_score"] = p

        for L, ns in enumerate(node_norm_list):
            df = pd.DataFrame({
                "layer": L,
                "node_index": np.arange(ns.size, dtype=int),
                "score": ns
            })
            p = os.path.join(results_dir, f"kan_node_score_L{L}.csv")
            df.to_csv(p, index=False)
            csv_paths[f"node_score_L{L}"] = p

        for L, es in enumerate(edge_norm_list):
            arr = np.asarray(es)
            if arr.ndim != 2:
                # reshape to (fan_in, fan_out) if needed
                arr = arr.reshape(arr.shape[0], -1)
            fin, fout = arr.shape
            df = pd.DataFrame({
                "layer": np.full(fin*fout, L, dtype=int),
                "in_idx": np.repeat(np.arange(fin, dtype=int), fout),
                "out_idx": np.tile(np.arange(fout, dtype=int), fin),
                "score": arr.ravel()
            })
            p = os.path.join(results_dir, f"kan_edge_score_L{L}.csv")
            df.to_csv(p, index=False)
            csv_paths[f"edge_score_L{L}"] = p

    # 7) Return everything
    return {
        "feature_score_raw": feat_score,
        "feature_score": feat_norm,
        "node_score_raw_list": node_scores,
        "node_score_list": node_norm_list,
        "edge_score_raw_list": edge_scores,
        "edge_score_list": edge_norm_list,
        "csv": csv_paths,
    }


model = best_models["s_n"]
res = compute_kan_built_in_relevance_full(
    model,
    X_scaled,
    results_dir="results",
    batch_size=2048,
    normalise=True,
    save_csv=True,
)

# Quick peek / top-k features (built-in relevance)
if res["feature_score"] is not None:
    fs = res["feature_score"].ravel()
    order = np.argsort(fs)[::-1][:10]
    print("\nTop 10 features (KAN built-in relevance):")
    for r, i in enumerate(order, 1):
        print(f"{r:2d}. x_{i:>3d}  score={fs[i]:.6f}")

# If node/edge lists are present, print basic info
if res["node_score_list"]:
    print("\nNode relevance per layer (lengths):", [len(v) for v in res["node_score_list"]])
if res["edge_score_list"]:
    shapes = []
    for arr in res["edge_score_list"]:
        a = np.asarray(arr)
        if a.ndim != 2:
            a = a.reshape(a.shape[0], -1)
        shapes.append(a.shape)
    print("Edge relevance per layer (fan_in × fan_out):", shapes)

print("\nCSV paths:", res["csv"])

In [ ]:
EDGE_THR = 1e-8
model = best_models['s_n']           # shorthand
# make sure the model has cache data set (use a representative batch of your inputs)
if getattr(model, "cache_data", None) is None:
    # X_train_tensor: the same scaled tensor you used for training (or a subset)
    model.set_cache_data(X_train_tensor)

# prune ONLY edges
model = model.prune(node_th=0.0, edge_th=EDGE_THR)

plt.figure(figsize=(12, 4))
model.plot(beta=10)
plt.savefig(os.path.join(RESULTS_DIR, f"{target_name}_KAN_training_plot_pruned_{EDGE_THR}.png"), dpi=900)
plt.show()

In [ ]:
state_snap = {k: v.detach().clone() for k, v in model.state_dict().items()}
# device = next(model.parameters()).device

In [ ]:
import torch
import numpy as np
import pandas as pd

def _to_np(x): return x.detach().cpu().numpy()

def rank_edges_and_nodes_from_state(state, topk=20):
    """
    state: model.state_dict() snapshot (as provided)
    Returns: (edges_df, nodes_df) ranked by relevance descending.
    """

    # ---------- layer 0: input -> hidden ----------
    C0 = state['act_fun.0.coef']        # spline coeffs
    S0b = state['act_fun.0.scale_base'] # scale base (amplitude-like)
    S0s = state['act_fun.0.scale_sp']   # scale sp (amplitude-like)
    M0 = state.get('act_fun.0.mask', None)

    # Try to standardize shapes to [H, D, K]
    # Your printout looked like [H, D, K], but we guard for [D, H, K].
    H0, D0, K0 = C0.shape if C0.shape[0] <= 64 else (C0.shape[1], C0.shape[0], C0.shape[2])
    if C0.shape != (H0, D0, K0):
        # transpose if stored as [D, H, K]
        C0 = C0.permute(1, 0, 2).contiguous()
        S0b = S0b.permute(1, 0).contiguous() if S0b.shape == (D0, H0) else S0b
        S0s = S0s.permute(1, 0).contiguous() if S0s.shape == (D0, H0) else S0s
        if M0 is not None and M0.shape == (D0, H0):
            M0 = M0.permute(1, 0).contiguous()

    # ensure masks present
    if M0 is None:
        M0 = torch.ones((H0, D0), dtype=C0.dtype, device=C0.device)

    # L1 of spline coeffs per edge (H,D)
    edge_power0 = C0.abs().sum(dim=2)  # [H,D]
    amp0 = S0b.abs() + S0s             # [H,D] (broadcast friendly)
    rel_edge0 = (edge_power0 * amp0 * M0).detach().cpu().numpy()  # [H,D]

    # Pack edges into table
    edge_rows = []
    for j in range(H0):
        for i in range(D0):
            edge_rows.append((i, j, rel_edge0[j, i]))
    edges_df = pd.DataFrame(edge_rows, columns=['in_idx', 'hid_idx', 'relevance'])
    edges_df.sort_values('relevance', ascending=False, inplace=True)

    # ---------- layer 1: hidden -> output ----------
    C1 = state['act_fun.1.coef']        # e.g., [H, K1] or [H, 1, K1]
    S1b = state['act_fun.1.scale_base'] # [H, 1] or [H]
    S1s = state['act_fun.1.scale_sp']   # [H, 1] or [H]
    M1 = state.get('act_fun.1.mask', None)

    # squeeze shapes to [H, K1]
    if C1.dim() == 3 and C1.shape[1] == 1:
        C1 = C1[:, 0, :]
    K1 = C1.shape[1]
    H1 = C1.shape[0]
    S1b = S1b.view(H1, -1)[:, 0]
    S1s = S1s.view(H1, -1)[:, 0]
    if M1 is None:
        M1 = torch.ones((H1,), dtype=C1.dtype, device=C1.device).view(H1)
    else:
        M1 = M1.view(H1, -1)[:, 0]

    node_out_strength = C1.abs().sum(dim=1) * (S1b.abs() + S1s) * M1  # [H]
    node_out_strength = _to_np(node_out_strength)

    # ---------- node relevance: (sum incoming) * outgoing ----------
    incoming_sum = edges_df.groupby('hid_idx')['relevance'].sum()
    incoming_sum = incoming_sum.reindex(range(H0)).fillna(0.0).values  # [H]
    node_rel = incoming_sum * node_out_strength[:H0]  # guard if H1==H0

    nodes_df = pd.DataFrame({
        'hid_idx': np.arange(H0, dtype=int),
        'incoming_sum': incoming_sum,
        'out_strength': node_out_strength[:H0],
        'relevance': node_rel
    }).sort_values('relevance', ascending=False)

    # ---------- top-k pretty print ----------
    print("\nTop nodes by relevance:")
    print(nodes_df.head(topk).to_string(index=False))

    print("\nTop edges (input->hidden) by relevance:")
    print(edges_df.head(topk).to_string(index=False))

    return edges_df, nodes_df

# Example usage (with your snapshot):
# edges_df, nodes_df = rank_edges_and_nodes_from_state(state_snap, topk=20)

## **Train 2**

In [ ]:
all_metrics = []
best_models = {}

for tgt in TARGETS:
    results, model = train_with_cv(X_scaled, y_dict[tgt], tgt, model=model1)
    all_metrics += results               # extend metrics list
    best_models[tgt] = model             # save model per target
    # y_scaled_last = y_scaled  # keep the scaled labels for this target

# save combined metrics
pd.DataFrame(all_metrics).to_csv(
    "results/KAN_all_cv_metrics_2.csv", index=False
)
model = best_models['s_p']
model2 = best_models['s_p']

In [ ]:
plt.figure(figsize=(12, 4))
model2.plot(beta=10)
plt.savefig(os.path.join(RESULTS_DIR, f"{target_name}_KAN_training_plot_2.png"), dpi=900)
plt.show()

In [ ]:
n_params = sum(p.numel() for p in model2.parameters())
print(f"Parameters: {n_params}")

model.plot()

In [ ]:
EDGE_THR = 1e-8
model = model.prune(edge_th=EDGE_THR)
plt.figure(figsize=(12, 4))
model2.plot(beta=10)
plt.savefig(os.path.join(RESULTS_DIR, f"{target_name}_KAN_training_plot_pruned_{EDGE_THR}.png"), dpi=900)
plt.show()

In [ ]:
edges2 = model2.auto_symbolic()

In [ ]:
model2.symbolic_formula()[:][:]


## **Train 3**

In [ ]:
for tgt in TARGETS:
    results, model = train_with_cv(X_scaled, y_dict[tgt], tgt, model=model2)
    all_metrics += results               # extend metrics list
    best_models[tgt] = model             # save model per target


save combined metrics
pd.DataFrame(all_metrics).to_csv("results/KAN_all_cv_metrics_3.csv", index=False)
model = best_models['s_p']
model3 = best_models['s_p']

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params}")

In [ ]:
EDGE_THR = 1e-6
model = model.prune(edge_th=EDGE_THR)
model.eval()
plt.figure(figsize=(12, 4))
model.plot(beta=10)
plt.savefig(os.path.join(RESULTS_DIR, f"{target_name}_KAN_training_plot_pruned_{EDGE_THR}.png"), dpi=900)
plt.show()

In [ ]:
n_params = sum(p.numel() for p in model3.parameters())
print(f"Parameters: {n_params}")

In [ ]:
edges3 = model3.auto_symbolic()

In [ ]:
model3.symbolic_formula()[:][:]


In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params}")

In [ ]:
EDGE_THR = 1e-4
model = model.prune(edge_th=EDGE_THR)
model.eval()
plt.figure(figsize=(12, 4))
model.plot(beta=10)
plt.savefig(os.path.join(RESULTS_DIR, f"{target_name}_KAN_training_plot_pruned_{EDGE_THR}.png"), dpi=900)
plt.show()

In [ ]:
edges = model.auto_symbolic(verbose=1, weight_simple=0.0, r2_threshold=0.0)




## **KAN attribution** 


In [ ]:
# ==== Imports ====
import os, math, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler

# ===========================
# =====  SETTINGS =======
# ===========================
DATA_PT       = "train_embeddings_250epochs_embdim128.pt"
RESULTS_DIR   = "results"
SAMPLES       = 15000
BATCH_SIZE    = 2048
TOP_K         = 16
FEATURE_NAMES = None   # list[str] or None → x_0..x_{D-1}

os.makedirs(RESULTS_DIR, exist_ok=True)


def _load_X_scaled(data_pt, samples):
    raw  = torch.load(data_pt).numpy()
    # last 3 are targets → keep rows with all > 0 (your ref rule)
    mask = (raw[:, -3:] > 0).all(axis=1)
    data = raw[mask]
    if samples is not None and samples < data.shape[0]:
        rng  = np.random.default_rng(42)
        idx  = rng.choice(data.shape[0], samples, replace=False)
        data = data[idx]
    X = data[:, :-3]

    # use the same scaler as training if present
    xsc_path = os.path.join(RESULTS_DIR, "X_scaler.pkl")
    if os.path.exists(xsc_path):
        X_scaler = joblib.load(xsc_path)
    else:
        X_scaler = StandardScaler().fit(X)
        joblib.dump(X_scaler, xsc_path)
    X_scaled = X_scaler.transform(X)
    return X_scaled

def _normalise(v):
    v = np.nan_to_num(v, nan=0.0, posinf=0.0, neginf=0.0)
    v = np.maximum(v, 0.0)
    s = v.sum()
    return (v / (s + 1e-12), s)

def _topk(feat_names, vals, k):
    D = len(vals)
    K = min(k, D)
    idx = np.argsort(vals)[::-1][:K]
    return [feat_names[i] for i in idx], vals[idx], idx


import matplotlib.pyplot as plt
import numpy as np


def _plot_barh(name, feat, vals, outfile):
    plt.figure(figsize=(10, 6))
    
    # Sort features and values together
    feat, vals = zip(*sorted(zip(feat, vals), key=lambda x: x[1]))

    # Bar plot
    bars = plt.barh(range(len(vals)), vals, color=plt.cm.plasma(np.linspace(0.2, 0.8, len(vals))))



    feat_latex = [f"$x_{{{f.split('_')[-1]}}}$" if f.startswith("x_") else f for f in feat]

    # Axis formatting
    plt.gca().invert_yaxis()
    plt.yticks(range(len(feat)), feat_latex, fontsize=16)
    plt.xticks(fontsize=14)
    plt.xlabel("Normalised score", fontsize=16)
    plt.title(name, fontsize=14, weight='bold', pad=10)

    # Grid
    plt.grid(axis='x', linestyle='--', alpha=0.6)

    # Remove right and top spines
    for spine in ['top', 'right']:
        plt.gca().spines[spine].set_visible(False)

    # Dynamic xlim
    xmax = float(max(vals)) if len(vals) else 1.0
    plt.xlim(0.006, xmax * 1.0)

    plt.tight_layout()
    plt.savefig(outfile, dpi=300)
    plt.show()



def _mean_abs_grad(model, X_t, batch=2048):
    device = X_t.device
    D = X_t.shape[1]
    gsum = torch.zeros(D, device=device)
    n = 0
    model.eval()
    for s in range(0, X_t.shape[0], batch):
        xb = X_t[s:s+batch].detach().clone().requires_grad_(True)
        yb = model(xb).view(-1)
        val = yb.sum()
        model.zero_grad(set_to_none=True)
        if xb.grad is not None:
            xb.grad.zero_()
        val.backward()
        g = xb.grad.detach().abs().mean(dim=0)
        # guard against NaNs
        g[torch.isnan(g)] = 0.0
        gsum += g
        n += 1
    return gsum / max(n, 1)

# ===========================
# ===== Main routine ========
# ===========================
def run_feature_importance(model,
                           data_pt=DATA_PT,
                           samples=SAMPLES,
                           batch_size=BATCH_SIZE,
                           top_k=TOP_K,
                           feature_names=FEATURE_NAMES):
    # --- prepare X ---
    X_scaled = _load_X_scaled(data_pt, samples)
    D = X_scaled.shape[1]
    if (feature_names is None) or (len(feature_names) != D):
        feature_names = [f"x_{i}" for i in range(D)]

    device = next(model.parameters()).device
    X_t = torch.tensor(X_scaled, dtype=torch.float32, device=device)

    # --- 1) Try KAN attribution ---
    used = "KAN attribution"
    score_norm = None
    try:
        model.eval()
        with torch.no_grad():
            for start in range(0, X_t.shape[0], batch_size):
                _ = model.get_act(X_t[start:start+batch_size])
        model.attribute()
        score = model.feature_score.detach().float().cpu().numpy().ravel()
        score_norm, ssum = _normalise(score)
        if ssum <= 0 or not np.isfinite(score_norm).any() or score_norm.max() == 0:
            raise RuntimeError("feature_score is zero/NaN")
    except Exception as e:
        print(f"[INFO] Attribution unavailable or empty → {e}")
        used = "Gradient sensitivity"
        # --- 2) Gradient fallback ---
        g = _mean_abs_grad(model, X_t, batch=batch_size).detach().cpu().numpy()
        score_norm, ssum = _normalise(g)

        # --- 3) Last resort: simple variance-based sensitivity ---
        if ssum <= 0 or score_norm.max() == 0:
            print("[INFO] Gradients are also (near) zero — using variance proxy.")
            # std across samples (on scaled space): larger std → potentially larger leverage
            v = X_scaled.std(axis=0)
            score_norm, ssum = _normalise(v)

    # --- report & plot ---
    print(f"\n=== Feature Scoring Method Used: {used} ===")
    print(f"D = {D}, L1 sum = {score_norm.sum():.6f}, max = {score_norm.max():.6g}")
    top_feat, top_vals, top_idx = _topk(feature_names, score_norm, top_k)
    print("Top 10 features:")
    for i, (n, v) in enumerate(zip(top_feat[:10], top_vals[:10]), 1):
        print(f" {i:2d}. {n:>12s}  {v:.6f}")

    # save CSV
    out_csv = os.path.join(RESULTS_DIR, "feature_importance_scores.csv")
    pd.DataFrame({"feature": feature_names, "score": score_norm}).to_csv(out_csv, index=False)
    print(f"\nSaved scores → {out_csv}")

    # plot TOP_K (guaranteed non-empty because we guard above)
    # title = f"Feature importance for Seebeck ({used})"
    title = f"Feature importance for Seebeck"
    out_png = os.path.join(RESULTS_DIR, "feature_importance_barh.png")
    _plot_barh(title, top_feat, top_vals, out_png)
    print(f"Saved plot   → {out_png}")

# ===========================
# ===== Run it ==============
# ===========================
# Assumes `model` is defined & loaded
run_feature_importance(model)